# 05 · Kill switch drill
Scopes **agent · user · global**. Order: ① gateway (APIM named values; optional subscription suspend) → ② identity (Okta app deactivate, AWS IAM deny) → ③ record (ServiceNow CI + incident) → platform-native manual steps (Foundry Control Plane, Entra Agent ID, Copilot Studio, Snowflake).
Without `authorize=True` every point writes a **DRY_RUN** receipt.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mccp").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

RUN_LIVE = False  # True only after deploying infra/mccp and filling .env (see docs/mccp/PLAN.md)

from mccp.config import Settings
from mccp.report import Console
settings = Settings.from_env(mode="azure" if RUN_LIVE else "local")
c = Console(color=False)
print("mode:", settings.mode, "(SIMULATED)" if settings.is_local else "(LIVE)")

In [ ]:
from mccp.demo import build_local_runtime
from mccp.killswitch import KillSwitch
from mccp.models import KillOrder
from mccp.receipts import ReceiptWriter
rt = build_local_runtime(settings)
ks = KillSwitch(settings, ReceiptWriter(settings.receipts_dir / "notebooks"), rt.simulator)
order = KillOrder(scope="agent", target="contoso-rev-cycle-coder", reason="drill", requested_by="ciso-office@contoso-health.example")

In [ ]:
for r in ks.execute(order, rt.agents, authorize=False):
    print(f"{r.status:<9} {r.enforcement_point:<17} {r.action}")

## Authorized (local: SIMULATED gateway change; external systems stay DRY_RUN)

In [ ]:
for r in ks.execute(order, rt.agents, authorize=True):
    print(f"{r.status:<9} {r.enforcement_point:<17} {r.action}")
print(rt.call("contoso-rev-cycle-coder", "p.nair@contoso-health.example", "hi").status, "<- 423 expected")

In [ ]:
order.revive = True
ks.execute(order, rt.agents, authorize=True)
print(rt.call("contoso-rev-cycle-coder", "p.nair@contoso-health.example", "hi").status, "<- 200 expected")

## Live (RUN_LIVE) — ARM PUT namedValues/killed-agents, Okta lifecycle, ServiceNow incident

In [ ]:
if RUN_LIVE:
    live_ks = KillSwitch(settings, ReceiptWriter(settings.receipts_dir))
    order.revive = False
    print([r.status for r in live_ks.execute(order, rt.agents, authorize=False)])  # set authorize=True deliberately
else:
    print("RUN_LIVE=False")